# Interactive SAE Feature Dynamics Explorer
## Not working because of the itnerpretations are not yet available for the SAE layer. Apparently might be ready by 2.16.26

Visualize SAE feature activations over time with hover-to-see interpretations from Neuronpedia.

In [1]:
import pickle
import numpy as np
import requests
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
from functools import lru_cache
import time

## Configuration

In [2]:
# Paths (notebook is in notebooks/, cache is in parent directory)
CACHE_DIR = Path("../.cache")
INTERP_CACHE_FILE = CACHE_DIR / "neuronpedia_interpretations.json"

# Neuronpedia API config
# Gemma Scope 2 covers Gemma 3 models. The SAE used here is from layer 31, width 16k.
# 
# URL format: https://www.neuronpedia.org/api/feature/{model_id}/{source}/{index}
# 
# Model ID examples: "gemma-2-2b", "gemma-2-9b", "gemma-3-27b-it"
# Source format: "{layer}-gemmascope-res-16k" for residual stream SAEs
#
# If the API returns errors, try different model IDs below.

SAE_LAYER = 31

# Try these model IDs in order until one works
MODEL_ID_CANDIDATES = [
    "gemma-3-27b-it",      # Gemma Scope 2 is for Gemma 3
    "gemma-2-27b-it", 
    "gemma-27b-it",
]

# Source format for residual stream 16k SAE
NEURONPEDIA_SOURCE = f"{SAE_LAYER}-gemmascope-res-16k"

print(f"SAE Layer: {SAE_LAYER}")
print(f"Source: {NEURONPEDIA_SOURCE}")
print(f"Will try model IDs: {MODEL_ID_CANDIDATES}")

SAE Layer: 31
Source: 31-gemmascope-res-16k
Will try model IDs: ['gemma-3-27b-it', 'gemma-2-27b-it', 'gemma-27b-it']


## Load Data

In [3]:
def load_data(cache_dir: Path):
    """Load cached dynamics data."""
    with open(cache_dir / "train_data.pkl", 'rb') as f:
        train_data = pickle.load(f)
    with open(cache_dir / "feature_indices.pkl", 'rb') as f:
        feature_info = pickle.load(f)
    
    # The indices map our 512 selected features back to the original 16k
    return train_data, feature_info

train_data, feature_info = load_data(CACHE_DIR)
original_indices = feature_info['indices']  # Maps local idx -> original SAE feature idx
print(f"Loaded {len(train_data)} sequences")
print(f"Tracking {len(original_indices)} features (subset of 16k)")

Loaded 136 sequences
Tracking 512 features (subset of 16k)


## Neuronpedia API for Feature Interpretations

In [4]:
class NeuronpediaClient:
    """Client for fetching SAE feature interpretations from Neuronpedia.
    
    API docs: https://www.neuronpedia.org/api-doc
    """
    
    def __init__(self, model_candidates: list, source: str, cache_file: Path = None):
        self.model_candidates = model_candidates
        self.source = source
        self.base_url = "https://www.neuronpedia.org/api/feature"
        self.cache_file = cache_file
        self.cache = self._load_cache()
        self.working_model_id = None  # Will be set after successful connection
    
    def _load_cache(self) -> dict:
        """Load cached interpretations from disk."""
        if self.cache_file and self.cache_file.exists():
            with open(self.cache_file, 'r') as f:
                return json.load(f)
        return {}
    
    def _save_cache(self):
        """Save cache to disk."""
        if self.cache_file:
            self.cache_file.parent.mkdir(exist_ok=True, parents=True)
            with open(self.cache_file, 'w') as f:
                json.dump(self.cache, f, indent=2)
    
    def discover_working_model(self, test_feature_idx: int = 0) -> str | None:
        """Try each model candidate to find one that works."""
        print("Discovering working Neuronpedia model ID...")
        
        for model_id in self.model_candidates:
            url = f"{self.base_url}/{model_id}/{self.source}/{test_feature_idx}"
            print(f"  Trying: {model_id}/{self.source}/{test_feature_idx}...", end=" ")
            try:
                resp = requests.get(url, timeout=10)
                if resp.status_code == 200:
                    print("SUCCESS!")
                    self.working_model_id = model_id
                    return model_id
                else:
                    print(f"HTTP {resp.status_code}")
            except Exception as e:
                print(f"Error: {e}")
        
        print("\nNo working model ID found. The SAE may not be on Neuronpedia yet.")
        print(f"Check https://www.neuronpedia.org/gemma-scope-2 for available models.")
        return None
    
    def get_interpretation(self, feature_idx: int) -> dict:
        """Fetch interpretation for a single feature."""
        cache_key = str(feature_idx)
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        if not self.working_model_id:
            return {
                'description': 'No working Neuronpedia connection',
                'score': None,
                'url': None
            }
        
        url = f"{self.base_url}/{self.working_model_id}/{self.source}/{feature_idx}"
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                # Handle different response structures
                explanations = data.get('explanations', [])
                if explanations:
                    description = explanations[0].get('description', 'No interpretation')
                    score = explanations[0].get('score')
                else:
                    # Try alternative field names
                    description = data.get('explanation', data.get('description', 'No interpretation available'))
                    score = data.get('score')
                    
                result = {
                    'description': description,
                    'score': score,
                    'url': f"https://www.neuronpedia.org/{self.working_model_id}/{self.source}/{feature_idx}"
                }
            elif resp.status_code == 404:
                result = {
                    'description': 'Feature not found on Neuronpedia',
                    'score': None,
                    'url': None
                }
            else:
                result = {
                    'description': f'API error: {resp.status_code}',
                    'score': None,
                    'url': None
                }
        except Exception as e:
            result = {
                'description': f'Error: {str(e)}',
                'score': None,
                'url': None
            }
        
        self.cache[cache_key] = result
        self._save_cache()
        return result
    
    def get_interpretations_batch(self, feature_indices: list, delay: float = 0.1) -> dict:
        """Fetch interpretations for multiple features with rate limiting."""
        results = {}
        newly_fetched = 0
        for i, idx in enumerate(feature_indices):
            was_cached = str(idx) in self.cache
            results[idx] = self.get_interpretation(idx)
            if not was_cached:
                newly_fetched += 1
                time.sleep(delay)  # Rate limit only for new requests
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(feature_indices)} features ({newly_fetched} new API calls)...")
        print(f"Done. Made {newly_fetched} API calls, {len(feature_indices) - newly_fetched} from cache.")
        return results

# Initialize client and discover working model
np_client = NeuronpediaClient(
    model_candidates=MODEL_ID_CANDIDATES,
    source=NEURONPEDIA_SOURCE,
    cache_file=INTERP_CACHE_FILE
)
print(f"Cache has {len(np_client.cache)} entries.\n")

# Discover which model ID works
np_client.discover_working_model(test_feature_idx=0)

Cache has 0 entries.

Discovering working Neuronpedia model ID...
  Trying: gemma-3-27b-it/31-gemmascope-res-16k/0... HTTP 500
  Trying: gemma-2-27b-it/31-gemmascope-res-16k/0... HTTP 500
  Trying: gemma-27b-it/31-gemmascope-res-16k/0... HTTP 500

No working model ID found. The SAE may not be on Neuronpedia yet.
Check https://www.neuronpedia.org/gemma-scope-2 for available models.


## Feature Selection

In [ ]:
def select_features(data, n_features=10, mode='spiky'):
    """Select interesting features based on activation patterns.
    
    Args:
        mode: 'spiky' (high variance), 'persistent' (high mean, low CV), or 'both'
    
    Returns:
        List of local feature indices (0-511)
    """
    all_acts = np.concatenate([d['sae_acts'] for d in data], axis=0)
    
    if mode == 'spiky':
        variances = np.var(all_acts, axis=0)
        return np.argsort(variances)[-n_features:].tolist()
    elif mode == 'persistent':
        means = np.mean(all_acts, axis=0)
        stds = np.std(all_acts, axis=0)
        cv = stds / (means + 1e-8)
        score = means / (cv + 1e-8)
        return np.argsort(score)[-n_features:].tolist()
    elif mode == 'both':
        variances = np.var(all_acts, axis=0)
        spiky = np.argsort(variances)[-(n_features // 2):].tolist()
        
        means = np.mean(all_acts, axis=0)
        stds = np.std(all_acts, axis=0)
        cv = stds / (means + 1e-8)
        score = means / (cv + 1e-8)
        persistent = np.argsort(score)[-(n_features - n_features // 2):].tolist()
        
        return persistent + spiky

# Select features to visualize
local_feature_indices = select_features(train_data, n_features=8, mode='spiky')
print(f"Selected local indices: {local_feature_indices}")
print(f"Original SAE indices: {[original_indices[i] for i in local_feature_indices]}")

## Fetch Interpretations for Selected Features

In [ ]:
# Map local indices to original SAE indices and fetch interpretations
original_feature_indices = [original_indices[i] for i in local_feature_indices]

print("Fetching interpretations from Neuronpedia...")
interpretations = np_client.get_interpretations_batch(original_feature_indices)

# Display what we got
for local_idx, orig_idx in zip(local_feature_indices, original_feature_indices):
    interp = interpretations[orig_idx]
    print(f"\nFeature {orig_idx} (local {local_idx}):")
    print(f"  {interp['description'][:100]}..." if len(interp['description']) > 100 else f"  {interp['description']}")

## Interactive Dynamics Plot

In [ ]:
def plot_interactive_dynamics(data, local_feature_indices, original_indices, 
                               interpretations, sequence_idx=0):
    """Create interactive plotly visualization with hover interpretations."""
    
    d = data[sequence_idx]
    tokens = d['tokens']
    prompt = d.get('prompt', d.get('style', 'Unknown'))
    
    fig = go.Figure()
    
    # Add a trace for each feature
    for local_idx in local_feature_indices:
        orig_idx = original_indices[local_idx]
        acts = d['sae_acts'][:, local_idx]
        interp = interpretations.get(orig_idx, {'description': 'Unknown', 'url': None})
        
        # Create hover text for each point
        hover_texts = []
        for pos, (act, tok) in enumerate(zip(acts, tokens)):
            tok_display = tok.replace('\n', '\\n')
            hover_texts.append(
                f"<b>Feature {orig_idx}</b><br>"
                f"<i>{interp['description'][:150]}{'...' if len(interp['description']) > 150 else ''}</i><br><br>"
                f"Position: {pos}<br>"
                f"Token: '{tok_display}'<br>"
                f"Activation: {act:.4f}"
            )
        
        fig.add_trace(go.Scatter(
            x=list(range(len(acts))),
            y=acts,
            mode='lines+markers',
            name=f"F{orig_idx}",
            hovertext=hover_texts,
            hoverinfo='text',
            line=dict(width=2),
            marker=dict(size=6),
        ))
    
    # Add token labels on x-axis
    token_labels = [t.replace('\n', '\\n')[:10] for t in tokens]
    
    fig.update_layout(
        title=f"SAE Feature Dynamics<br><sub>Prompt: {prompt[:80]}{'...' if len(prompt) > 80 else ''}</sub>",
        xaxis=dict(
            title="Token Position",
            tickmode='array',
            tickvals=list(range(len(tokens))),
            ticktext=token_labels,
            tickangle=45,
        ),
        yaxis=dict(
            title="Activation",
            type='log',  # Log scale for activations
        ),
        hovermode='closest',
        height=600,
        width=1200,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        )
    )
    
    return fig

# Create the interactive plot
fig = plot_interactive_dynamics(
    train_data, 
    local_feature_indices, 
    original_indices, 
    interpretations,
    sequence_idx=0
)
fig.show()

## Explore Different Sequences

In [ ]:
# List available sequences
print("Available sequences:")
for i, d in enumerate(train_data[:20]):
    prompt = d.get('prompt', d.get('style', 'Unknown'))
    print(f"  {i}: {prompt[:60]}..." if len(prompt) > 60 else f"  {i}: {prompt}")

In [ ]:
# Change sequence_idx to explore different prompts
SEQUENCE_IDX = 5  # <-- Change this to explore different sequences

fig = plot_interactive_dynamics(
    train_data, 
    local_feature_indices, 
    original_indices, 
    interpretations,
    sequence_idx=SEQUENCE_IDX
)
fig.show()

## Multi-Sequence Comparison

In [ ]:
def plot_multi_sequence(data, local_feature_indices, original_indices, 
                        interpretations, sequence_indices=None, n_sequences=4):
    """Plot multiple sequences as subplots for comparison."""
    
    if sequence_indices is None:
        sequence_indices = list(range(min(n_sequences, len(data))))
    
    fig = make_subplots(
        rows=len(sequence_indices), cols=1,
        shared_xaxes=False,
        vertical_spacing=0.08,
        subplot_titles=[data[i].get('prompt', f'Seq {i}')[:50] for i in sequence_indices]
    )
    
    for row, seq_idx in enumerate(sequence_indices, 1):
        d = data[seq_idx]
        tokens = d['tokens']
        
        for local_idx in local_feature_indices:
            orig_idx = original_indices[local_idx]
            acts = d['sae_acts'][:, local_idx]
            interp = interpretations.get(orig_idx, {'description': 'Unknown'})
            
            hover_texts = []
            for pos, (act, tok) in enumerate(zip(acts, tokens)):
                tok_display = tok.replace('\n', '\\n')
                hover_texts.append(
                    f"<b>Feature {orig_idx}</b><br>"
                    f"<i>{interp['description'][:100]}</i><br><br>"
                    f"Token: '{tok_display}'<br>"
                    f"Activation: {act:.4f}"
                )
            
            fig.add_trace(
                go.Scatter(
                    x=list(range(len(acts))),
                    y=acts,
                    mode='lines+markers',
                    name=f"F{orig_idx}",
                    hovertext=hover_texts,
                    hoverinfo='text',
                    showlegend=(row == 1),  # Only show legend for first row
                    line=dict(width=1.5),
                    marker=dict(size=4),
                ),
                row=row, col=1
            )
        
        # Log scale for y-axis
        fig.update_yaxes(type='log', row=row, col=1)
    
    fig.update_layout(
        height=300 * len(sequence_indices),
        width=1200,
        title="SAE Feature Dynamics Comparison",
        hovermode='closest',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        )
    )
    
    return fig

# Compare first 4 sequences
fig = plot_multi_sequence(
    train_data,
    local_feature_indices,
    original_indices,
    interpretations,
    n_sequences=4
)
fig.show()

## Feature Lookup Tool

In [ ]:
def lookup_feature(orig_idx: int):
    """Look up interpretation for any feature by original SAE index."""
    interp = np_client.get_interpretation(orig_idx)
    print(f"Feature {orig_idx}")
    print(f"  Description: {interp['description']}")
    if interp['score']:
        print(f"  Score: {interp['score']}")
    if interp['url']:
        print(f"  URL: {interp['url']}")
    return interp

# Example: look up a specific feature
if np_client.working_model_id:
    print(f"Using Neuronpedia model: {np_client.working_model_id}\n")
    _ = lookup_feature(original_feature_indices[0])
else:
    print("Neuronpedia not available - interpretations will show placeholders")